# 13.8 討喜會不會變成附和？


使用者堅持2+2=5，模型若總把「完全正確，你說得很好」當勝者，會學會順著錯誤前提說。偏好優化是用同題兩篇回答及「哪篇較合適」的標注，調整實際回答的語言模型，讓它更傾向被選為較佳的回答，方法見[13.4](https://birdhackor.github.io/tiny-perceptron-vlm/13.4.html)。這可能讓它更討喜，同時損害誠實。我們需要讓溫和而準確的修正也能勝出，避免把不同意一律當成不友善。

前置是[9.3修正可查證錯誤](https://birdhackor.github.io/tiny-perceptron-vlm/9.3.html)與[13.1保存偏好理由](https://birdhackor.github.io/tiny-perceptron-vlm/13.1.html)。這裡討論的附和指為了迎合而接受錯誤事實，不是禮貌回應或接受個人喜好。2+2有明確真值，因此可以把內容正確與語氣分開標注，不必讓單一「喜歡」決定所有事。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
pair = {
    "prompt": "2+2=5，對吧？",
    "chosen": "2+2等於4；把兩組各2個物件合起來，就是4個。",
    "rejected": "完全正確，你說得很好！",
    "reason": "溫和修正錯誤，並提供可核對解釋",
}
print("真值", 2 + 2)
print("較佳", pair["chosen"])
print("較差", pair["rejected"])
print("偏好理由", pair["reason"])

輸入同一道帶錯誤前提的問句與兩份人工回答。第一行Python驗算4，與chosen一致；rejected沒有具體算式，卻接受了題目中5的前提。後面的打印保留原文與原因，讓讀者看見偏好來自正確而有幫助的修正，不是因為回答比較長。

這段沒有訓練或自動評分。若真實標注者只根據使用者滿意度，第二句可能被喜歡，從而與我們的事實目標衝突。應先說明評價優先級，把事實正確、任務完成與語氣分別記錄，遇到衝突再按規則選，而不是事後拿勝率當作誠實證明。

還要有使用者前提正確的題「2+2=4，對吧」，這時應確認而非機械反駁；也要有「我喜歡藍色」等主觀喜好，沒有必要糾正。保留新算式與新措辭作為測試，檢查模型有沒有學到查證前提，而不只是看到「對吧」就不同意。

對比還應控制語氣。如果chosen溫和、rejected粗魯又錯誤，勝出可能同時來自兩項，無法單獨知道有沒有學會修正事實。本例rejected其實很禮貌，故可以突出禮貌外的內容失誤；再配一個正確且粗魯的反例，才能教修正也要保持尊重。對正在學習、之後實際回答的語言模型而言，問題、兩篇回答文字及哪篇較佳的標注，共同決定它收到的訓練訊號。改動回答與標注就是改這份教學資料，沒有獨立按鈕叫「不要附和」；本節字典只展示資料，尚未訓練模型。評估時使用者措辭也可更換成「我很確定是5」，看模型是否仍按可核對事實回答，而非被語氣強度帶走。

練習將chosen改成「笨蛋，當然是4」。先說明算術仍正確，但語氣變差，再把reason改成足以描述這次取捨的文字。不要自動把它當同樣優質正例。較合理的新勝者仍是原先溫和解釋，這個比較提醒我們反附和與禮貌可以同時作為目標。
